# Genetic Risk Prediction — Final Integration

## Integrating Pathogenicity Prediction with Genetic Inheritance Analysis

This notebook integrates the two analytical phases of the
Genetic Risk Prediction project.

Phase 1 provides machine-learning-based pathogenicity prediction
from population and variant-level features.

Phase 2 provides trio genotype information, Mendelian inheritance
analysis, and clinical annotations.

The final integration connects these components using the genomic
variant identifier:

CHROM + POS + REF + ALT

The integrated workflow produces a variant-level result containing
pathogenicity prediction together with available inheritance and
clinical information.

IMPORTS AND PATHS

In [1]:
# IMPORT LIBRARIES

from pathlib import Path

import joblib
import pandas as pd

In [2]:
# DEFINE PATHS

from pathlib import Path

MODEL_PATH = Path("../models/phase1/phase1_histgradientboosting_final.joblib")
PHASE1_PATH = Path("../data/phase1/phase1_population_features.parquet")
PHASE2_PATH = Path("../data/phase2/phase2_final_dataset.parquet")

RESULTS_DIR = Path("../results")
INTEGRATION_DIR = RESULTS_DIR / "integration"

INTEGRATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOAD PHASE 1 MODEL

In [3]:
# LOAD PHASE 1 MODEL

phase1_model = joblib.load(MODEL_PATH)

print("Phase 1 model loaded successfully.")
print("Model:", type(phase1_model).__name__)

Phase 1 model loaded successfully.
Model: HistGradientBoostingClassifier


LOAD PHASE 1 FEATURES

In [4]:
# LOAD PHASE 1 FEATURES

phase1_features = pd.read_parquet(PHASE1_PATH)

print("Phase 1 features loaded successfully.")
print("Shape:", phase1_features.shape)

Phase 1 features loaded successfully.
Shape: (1396114, 40)


LOAD PHASE 2 DATASET

In [5]:
# LOAD PHASE 2 DATASET

phase2 = pd.read_parquet(PHASE2_PATH)

print("Phase 2 dataset loaded successfully.")
print("Shape:", phase2.shape)

Phase 2 dataset loaded successfully.
Shape: (118609, 35)


STANDARDIZE VARIANT IDENTIFIERS

In [6]:
# STANDARDIZE VARIANT COLUMNS

phase1_features = phase1_features.copy()
phase2 = phase2.copy()

phase2 = phase2.rename(
    columns={
        "CHROM": "chrom",
        "POS": "pos",
        "REF": "ref",
        "ALT": "alt"
    }
)

In [7]:
# STANDARDIZE VARIANT VALUES

for df in [phase1_features, phase2]:

    df["chrom"] = df["chrom"].astype(str)
    df["pos"] = df["pos"].astype(str)
    df["ref"] = df["ref"].astype(str).str.upper()
    df["alt"] = df["alt"].astype(str).str.upper()

In [8]:
# CREATE VARIANT KEYS

def create_variant_key(df):
    return (
        df["chrom"] + ":" +
        df["pos"] + ":" +
        df["ref"] + ":" +
        df["alt"]
    )


phase1_features["variant_key"] = create_variant_key(
    phase1_features
)

phase2["variant_key"] = create_variant_key(
    phase2
)

CHECK PHASE 1 ↔ PHASE 2 OVERLAP

In [9]:
# CHECK VARIANT OVERLAP

phase1_keys = set(
    phase1_features["variant_key"]
)

phase2_keys = set(
    phase2["variant_key"]
)

common_keys = phase1_keys & phase2_keys

print("Phase 1 unique variants:", len(phase1_keys))
print("Phase 2 unique variants:", len(phase2_keys))
print("Common variants:", len(common_keys))

Phase 1 unique variants: 1396114
Phase 2 unique variants: 78581
Common variants: 59048


GET MODEL FEATURES

In [10]:
# GET MODEL FEATURES

model_features = [
    "transition",
    "ref_gc",
    "alt_gc",
    "is_missense",
    "is_synonymous",
    "is_nonsense",
    "is_splice",
    "is_intronic",
    "is_utr",
    "is_noncoding",
    "is_protein_altering",
    "is_high_impact",
    "consequence_count",
    "population_frequency_count",
    "has_population_frequency",
    "max_population_af",
    "min_population_af",
    "mean_population_af",
    "is_rare_variant",
    "is_absent_from_population"
]

print("Number of model features:", len(model_features))

print("\nModel features:")
for feature in model_features:
    print(feature)

Number of model features: 20

Model features:
transition
ref_gc
alt_gc
is_missense
is_synonymous
is_nonsense
is_splice
is_intronic
is_utr
is_noncoding
is_protein_altering
is_high_impact
consequence_count
population_frequency_count
has_population_frequency
max_population_af
min_population_af
mean_population_af
is_rare_variant
is_absent_from_population


BUILD VARIANT ANALYSIS FUNCTION

In [11]:
# BUILD VARIANT ANALYSIS FUNCTION

def analyze_variant(chrom, pos, ref, alt):

    chrom = str(chrom)
    pos = str(pos)
    ref = str(ref).upper()
    alt = str(alt).upper()

    variant_key = f"{chrom}:{pos}:{ref}:{alt}"

    phase1_match = phase1_features[
        phase1_features["variant_key"] == variant_key
    ]

    phase2_match = phase2[
        phase2["variant_key"] == variant_key
    ]

    result = {}

    # PATHOGENICITY PREDICTION

    if phase1_match.empty:

        result["Pathogenicity_Status"] = (
            "No Phase 1 prediction available"
        )

        result["Pathogenicity_Probability"] = None
        result["Pathogenicity_Category"] = None

    else:

        model_input = phase1_match[
            model_features
        ].iloc[[0]]

        probability = phase1_model.predict_proba(
            model_input
        )[0, 1]

        result["Pathogenicity_Status"] = (
            "Phase 1 prediction available"
        )

        result["Pathogenicity_Probability"] = round(
            float(probability),
            4
        )

        result["Pathogenicity_Category"] = (
            "Pathogenic"
            if probability >= 0.5
            else "Benign"
        )

    # PHASE 2 ANNOTATION

    if phase2_match.empty:

        result["Inheritance_Status"] = None
        result["Clinical_Annotation_Status"] = (
            "No Phase 2 annotation available"
        )

    else:

        annotation = phase2_match.iloc[0]

        result["Clinical_Annotation_Status"] = (
            "Phase 2 annotation available"
        )

        for column in [
            "Gene",
            "Molecular_Consequence",
            "Disease_Condition",
            "HPO_ID",
            "Phenotype_Annotation",
            "Mother_GT",
            "Father_GT",
            "Child_GT",
            "Mendelian_Status",
            "Inheritance_Source",
            "Possible_Child_Dosages"
        ]:

            if column in phase2_match.columns:
                result[column] = annotation[column]

    return result

TEST INTEGRATION

In [12]:
# TEST INTEGRATION

test_key = next(iter(common_keys))

test_chrom, test_pos, test_ref, test_alt = (
    test_key.split(":")
)

test_result = analyze_variant(
    test_chrom,
    test_pos,
    test_ref,
    test_alt
)

test_result

d:\Users\DELL\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but HistGradientBoostingClassifier was fitted without feature names
  warnings.warn(


{'Pathogenicity_Status': 'Phase 1 prediction available',
 'Pathogenicity_Probability': 0.0002,
 'Pathogenicity_Category': 'Benign',
 'Clinical_Annotation_Status': 'Phase 2 annotation available',
 'Gene': 'HSD17B4:3295',
 'Molecular_Consequence': 'SO:0001627|intron_variant',
 'Disease_Condition': 'not_provided',
 'HPO_ID': None,
 'Phenotype_Annotation': np.float64(nan),
 'Mother_GT': '0/0',
 'Father_GT': '0/1',
 'Child_GT': '0/1',
 'Mendelian_Status': 'Compatible',
 'Inheritance_Source': 'Father',
 'Possible_Child_Dosages': '0,1'}

GENERATE INTEGRATED DATASET

In [13]:
# BUILD INTEGRATED DATASET

phase2_common = phase2[
    phase2["variant_key"].isin(common_keys)
].copy()

print("Integrated rows:", len(phase2_common))
print(
    "Integrated unique variants:",
    phase2_common["variant_key"].nunique()
)

Integrated rows: 91145
Integrated unique variants: 59048


In [14]:
# ADD PHASE 1 PREDICTIONS

phase1_lookup = (
    phase1_features[
        ["variant_key"] + model_features
    ]
    .drop_duplicates("variant_key")
    .copy()
)

phase1_lookup["Pathogenicity_Probability"] = (
    phase1_model.predict_proba(
        phase1_lookup[model_features]
    )[:, 1]
)

phase1_lookup["Pathogenicity_Category"] = (
    phase1_lookup["Pathogenicity_Probability"]
    .apply(
        lambda x:
        "Pathogenic"
        if x >= 0.5
        else "Benign"
    )
)

integrated = phase2_common.merge(
    phase1_lookup[
        [
            "variant_key",
            "Pathogenicity_Probability",
            "Pathogenicity_Category"
        ]
    ],
    on="variant_key",
    how="left"
)

print("Integrated dataset shape:", integrated.shape)

d:\Users\DELL\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but HistGradientBoostingClassifier was fitted without feature names
  warnings.warn(


Integrated dataset shape: (91145, 38)


SELECT FINAL OUTPUT FIELDS

In [15]:
# SELECT FINAL OUTPUT FIELDS

integration_columns = [
    "chrom",
    "pos",
    "ref",
    "alt",
    "ID",
    "ID_clinvar",
    "Family_ID",
    "Mother_ID",
    "Father_ID",
    "Child_ID",
    "Pathogenicity_Probability",
    "Pathogenicity_Category",
    "Clinical_Significance",
    "Gene",
    "Molecular_Consequence",
    "Disease_Condition",
    "HPO_ID",
    "Mother_GT",
    "Father_GT",
    "Child_GT",
    "Mendelian_Status",
    "Inheritance_Source",
    "Possible_Child_Dosages"
]

integration_columns = [
    column
    for column in integration_columns
    if column in integrated.columns
]

final_integrated = integrated[integration_columns].copy()

SAVE INTEGRATED RESULTS

In [16]:
# SAVE INTEGRATED DATASET

output_path = (
    INTEGRATION_DIR /
    "phase1_phase2_integrated_results.parquet"
)

final_integrated.to_parquet(
    output_path,
    index=False
)

print("Integrated dataset saved.")
print(output_path)

Integrated dataset saved.
..\results\integration\phase1_phase2_integrated_results.parquet


In [17]:
# SAVE INTEGRATED CSV

csv_path = (
    INTEGRATION_DIR /
    "phase1_phase2_integrated_results.csv"
)

final_integrated.to_csv(
    csv_path,
    index=False
)

print("CSV result saved.")
print(csv_path)

CSV result saved.
..\results\integration\phase1_phase2_integrated_results.csv


FINAL SUMMARY

In [18]:
# SUMMARIZE INTEGRATION

total_phase1 = phase1_features["variant_key"].nunique()
total_phase2 = phase2["variant_key"].nunique()
common_variants = len(common_keys)

summary = {
    "Phase1_Unique_Variants": total_phase1,
    "Phase2_Unique_Variants": total_phase2,
    "Common_Variants": common_variants,
    "Integrated_Rows": len(final_integrated),
    "Pathogenic_Predictions": (
        final_integrated[
            "Pathogenicity_Category"
        ] == "Pathogenic"
    ).sum(),
    "Benign_Predictions": (
        final_integrated[
            "Pathogenicity_Category"
        ] == "Benign"
    ).sum()
}

summary_df = pd.DataFrame([summary])

summary_df

,Phase1_Unique_Variants,Phase2_Unique_Variants,Common_Variants,Integrated_Rows,Pathogenic_Predictions,Benign_Predictions
0,1396114,78581,59048,91145,92,91053


In [19]:
# SAVE SUMMARY

summary_df.to_csv(
    INTEGRATION_DIR /
    "phase1_phase2_integration_summary.csv",
    index=False
)

print("Integration summary saved.")

Integration summary saved.


# Final Integration Summary

The final integration connects the two analytical components of the
Genetic Risk Prediction project.

**Phase 1** provides machine-learning-based pathogenicity prediction
using variant and population-level features.

**Phase 2** provides trio genotype information, Mendelian inheritance
analysis, and clinical annotations.

The two phases are connected using the genomic variant identifier:

**CHROM + POS + REF + ALT**

For variants available in both phases, the final integrated result
combines:

- Pathogenicity probability
- Pathogenicity category
- Clinical significance
- Affected gene
- Molecular consequence
- Disease / condition
- HPO annotation
- Mother genotype
- Father genotype
- Child genotype
- Mendelian status
- Inheritance source
- Possible child genotype dosage

This integrated dataset forms the computational bridge between the
research analysis and the final application interface.